# Proof of Concept – Instant RAG (ADD/MUL als Proxy-Tools)

Bachelorarbeit Roland Kummer (HS Offenburg × t2b AG). Dieses Notebook dokumentiert alle PoC-Läufe gegen den
unveränderten Referenz-Harness `src/instantrag_qwen_demo.py`. Plan, Entscheidungen und Status stehen in
`doc/poc-sprint-plan.md`.

**Regeln für dieses Notebook**
- Der Harness wird nur importiert und von außen beobachtet, nie verändert.
- Rohdaten sind append-only (`results/runs.jsonl`); der Offline-Selbsttest schreibt nach `results/selftest.jsonl`.
- `model_cfg` wird nie ausgegeben (enthält den API-Key), Request-Header werden nie geloggt.
- Zellen, die Requests an das LLM auslösen, stehen standardmäßig auf `RUN = False`.

## 0. Zweck, Hypothesen, fixierte Schwellen

*Gliederungspunkt – wird vor E1 ausgearbeitet.* Hier stehen Zweck des PoC, die Hypothesen H1–H9
(Plan §6) und die am 15.11.2026 entschiedenen Erfolgsschwellen für H1 (Plan §2, O6) mit Datum.

## 1. Einrichtung

Pfade, Import des Harness, Konfiguration (ohne Key), Metadaten, JSONL-Writer, Rekorder, Darstellung und
Offline-Selbsttest. Prompt-Varianten S1/S2 und die Aufgabenliste folgen in Schritt 5.

### 1.1 Pfade

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    """Walk up from the working directory until harness and system prompt are found."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_harness = (candidate / "src" / "instantrag_qwen_demo.py").is_file()
        has_prompt = (candidate / "config" / "instantrag.txt").is_file()
        if has_harness and has_prompt:
            return candidate
    raise FileNotFoundError(f"repository root not found above {start}")


ROOT = find_repo_root()
SRC_DIR = ROOT / "src"
SETTINGS_PATH = ROOT / "config" / "llm_config.json"
RESULTS_DIR = ROOT / "results"
RUNS_PATH = RESULTS_DIR / "runs.jsonl"          # experiment data, append-only
SELFTEST_PATH = RESULTS_DIR / "selftest.jsonl"  # offline self-test only
PLAYGROUND_PATH = RESULTS_DIR / "playground.jsonl"  # section 8, exploratory runs only
print("repo root:", ROOT.name)

### 1.2 Import des Harness

In [ ]:
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import instantrag_qwen_demo as harness
from llm_request import FatalConfigError

assert Path(harness.__file__).resolve().parent == SRC_DIR, "harness imported from an unexpected location"
print("python :", sys.version.split()[0])
print("harness:", Path(harness.__file__).relative_to(ROOT).as_posix())

### 1.3 Konfiguration (ohne Key)

`harness.load_model()` legt den API-Key in `model_cfg["token"]` ab. Deshalb wird `model_cfg` in ein `dict` mit
geschwärzter Darstellung verpackt; angezeigt wird nur eine Whitelist. `temperature` und `max_tokens` aus
`config/llm_config.json` ignoriert der Harness – maßgeblich sind die Werte hier (Plan §2, §7).

In [ ]:
REFERENCE_MODEL = "qwen-3.5-35b"  # M1 (plan O3)
TEMPERATURE = 0.1
MAX_TOKENS = 2048
PUBLIC_CFG_KEYS = ("model", "endpoint")


class SafeCfg(dict):
    """model_cfg as the harness expects it; its repr never shows the content (API key)."""

    def __repr__(self):
        return f"<SafeCfg model={self.get('model')!r} endpoint={self.get('endpoint')!r} token=***>"

    __str__ = __repr__

    def _repr_pretty_(self, printer, cycle):
        printer.text(repr(self))


def load_cfg(model=None):
    """Load the model config through the unmodified harness; optionally set the model."""
    cfg = SafeCfg(harness.load_model(SETTINGS_PATH))
    if model:
        cfg["model"] = model
    return cfg


def public_cfg(cfg):
    """The only part of the config that may be displayed or logged."""
    return {key: cfg.get(key) for key in PUBLIC_CFG_KEYS}


try:
    model_cfg = load_cfg(REFERENCE_MODEL)
    print(public_cfg(model_cfg))
except FatalConfigError as error:
    # no key on this machine: offline cells still work, live runs do not
    model_cfg = None
    print("no API key available:", error)

### 1.4 Metadaten und Prompt-Versionen

Die Prompt-Prüfsumme wird über den **geladenen Text** gebildet (wie `load_system_prompt()` ihn liefert), nicht
über die Dateibytes – Zeilenenden können sich zwischen Rechnern unterscheiden. Dasselbe gilt für
`harness_sha256`. `git_dirty` meldet geänderte *getrackte* Dateien, `git_dirty_files` listet sie.

In [ ]:
import hashlib
import subprocess


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _git(*args):
    try:
        done = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True)
        # rstrip only: porcelain lines start with a status column that may be a space
        return done.stdout.rstrip()
    except (OSError, subprocess.CalledProcessError):
        return None


def git_commit():
    return _git("rev-parse", "HEAD")


def git_dirty_files():
    """Tracked files that differ from the commit (sorted); None if git is unavailable."""
    status = _git("status", "--porcelain", "--untracked-files=no")
    if status is None:
        return None
    # porcelain v1: two status letters, a space, then the path ("old -> new" for renames)
    return sorted(line[3:] for line in status.splitlines() if line.strip())


def git_dirty():
    """True if tracked files differ from the commit; None if git is unavailable."""
    files = git_dirty_files()
    return None if files is None else bool(files)


def harness_sha256():
    return sha256_text(Path(harness.__file__).read_text(encoding="utf-8"))


PROMPT_PATHS = {
    "S0": ROOT / "config" / "instantrag.txt",  # S1/S2 follow in step 5
}


def load_prompt(version):
    path = PROMPT_PATHS[version]
    text = harness.load_system_prompt(path)
    return {
        "version": version,
        "path": path.relative_to(ROOT).as_posix(),
        "text": text,
        "sha256": sha256_text(text),
        "chars": len(text),
    }


print("git commit    :", git_commit(), "| tracked files modified:", git_dirty_files())
print("harness sha256:", harness_sha256())
for version in PROMPT_PATHS:
    prompt = load_prompt(version)
    print(f"prompt {version}     : {prompt['path']} | {prompt['chars']} chars | sha256 {prompt['sha256']}")

**Volltext S0** (`config/instantrag.txt`, unverändert):

In [ ]:
print(load_prompt("S0")["text"])

### 1.5 JSONL-Writer (append-only)

Eine JSON-Zeile pro Lauf, mit `schema_version`. Es wird nur angehängt, nie überschrieben oder gefiltert.
Enthält eine Zeile den API-Key, wird nicht geschrieben.

In [ ]:
import json

SCHEMA_VERSION = 1


def append_jsonl(path, record, secrets=()):
    """Append one record as one line. Refuses to write if a secret appears in the line."""
    line = json.dumps(record, ensure_ascii=False, default=str)
    for secret in secrets:
        if secret and secret in line:
            raise ValueError("refusing to write: the record contains a secret")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8", newline="\n") as handle:
        handle.write(line + "\n")


def read_jsonl(path):
    """Read all records, unfiltered. A missing file is an empty list."""
    path = Path(path)
    if not path.is_file():
        return []
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

### 1.6 Rekorder um `run_turn` und protokollierter Lauf

`run_logged()` ersetzt `harness.run_turn` nur für die Dauer eines Laufs durch einen Rekorder, der das Original
aufruft und Eingabe-Messages, Parameter, Rückgabe und Zeiten mitschreibt. `run_phase` läuft unverändert, seine
Ausgabe wird als `harness_log` gespeichert. `seed` geht über `PAYLOAD_EXTRAS` in den Request. Auch ein Lauf,
der mit einer Exception endet, wird geschrieben (Feld `error`).

In [ ]:
import contextlib
import copy
import io
import time
import uuid
from datetime import datetime, timezone
from unittest import mock


def _utc_now():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def _error_info(error):
    return {"type": type(error).__name__, "message": str(error)}


def make_recorder(original_run_turn, turns):
    """Wrap run_turn: call the original and record input, parameters, result and timing."""

    def recording_run_turn(model_cfg, messages, max_tokens, temperature, stop_at_tool, dep_value=None):
        entry = {
            "stage": len(turns) + 1,
            "started_at": _utc_now(),
            "messages": copy.deepcopy(messages),
            "params": {
                "max_tokens": max_tokens,
                "temperature": temperature,
                "stop_at_tool": stop_at_tool,
                "dep_value": dep_value,
            },
            "payload_extras": copy.deepcopy(harness.PAYLOAD_EXTRAS),
            "result": None,
            "error": None,
            "duration_s": None,
        }
        turns.append(entry)
        started = time.perf_counter()
        try:
            result = original_run_turn(
                model_cfg, messages, max_tokens, temperature, stop_at_tool=stop_at_tool, dep_value=dep_value
            )
            entry["result"] = copy.deepcopy(result)
            return result
        except Exception as error:
            entry["error"] = _error_info(error)
            raise
        finally:
            entry["duration_s"] = round(time.perf_counter() - started, 3)

    return recording_run_turn


def run_logged(model_cfg, prompt, question, phase, *, experiment, seed, out_path,
               max_tokens=MAX_TOKENS, temperature=TEMPERATURE, expected=None, note=None):
    """Run one phase through the unmodified run_phase and append one record to out_path."""
    turns = []
    record = {
        "schema_version": SCHEMA_VERSION,
        "run_id": datetime.now().strftime("%Y%m%dT%H%M%S") + "-" + uuid.uuid4().hex[:6],
        "started_at": _utc_now(),
        "experiment": experiment,
        "phase": phase,
        **public_cfg(model_cfg),
        "temperature": temperature,
        "max_tokens": max_tokens,
        "seed": seed,
        "prompt_version": prompt["version"],
        "prompt_path": prompt["path"],
        "prompt_sha256": prompt["sha256"],
        "prompt_chars": prompt["chars"],
        "git_commit": git_commit(),
        "git_dirty": git_dirty(),
        "git_dirty_files": git_dirty_files(),
        "harness_sha256": harness_sha256(),
        "question": question,
        "expected": expected,
        "note": note,
        "turns": turns,
        "harness_log": None,
        "error": None,
        "duration_s": None,
    }
    extras = {} if seed is None else {"seed": seed}
    log = io.StringIO()
    started = time.perf_counter()
    try:
        with mock.patch.object(harness, "run_turn", make_recorder(harness.run_turn, turns)), \
                mock.patch.object(harness, "PAYLOAD_EXTRAS", extras), \
                contextlib.redirect_stdout(log):
            harness.run_phase(
                model_cfg=model_cfg,
                system_prompt=prompt["text"],
                question=question,
                phase=phase,
                max_tokens=max_tokens,
                temperature=temperature,
            )
    except Exception as error:
        # infrastructure errors are data too: keep the run, report it separately
        record["error"] = _error_info(error)
    record["duration_s"] = round(time.perf_counter() - started, 3)
    record["harness_log"] = log.getvalue()
    append_jsonl(out_path, record, secrets=[model_cfg.get("token")])
    if record["error"]:
        print(f"run {record['run_id']} ended with an error: {record['error']}")
    return record

### 1.7 Darstellung eines Laufs

Pro Stage: gesendete Nachrichten unverändert (der Systemprompt nur als Version + Prüfsumme, Volltext siehe 1.4),
dann die Ausgabe. Der Reasoning-Kanal ist als „Thinking“ beschriftet. Die Id eines abgefangenen Tags wird mit
`harness.assign_id` nachvollzogen (dieselbe Logik wie in `run_phase`).

In [ ]:
import html

from IPython.display import HTML, display

PRE_STYLE = "white-space:pre-wrap;border:1px solid rgba(128,128,128,.5);padding:6px;margin:2px 0 10px 0"


def _pre(text):
    if not text:
        return "<p><i>(leer)</i></p>"
    return f'<pre style="{PRE_STYLE}">{html.escape(text)}</pre>'


def _table(rows):
    cells = "".join(
        f'<tr><th style="text-align:left">{html.escape(str(key))}</th>'
        f'<td style="text-align:left">{html.escape(str(value))}</td></tr>'
        for key, value in rows
    )
    return f"<table>{cells}</table>"


def message_label(index, message):
    role, content = message["role"], message["content"]
    if role == "system":
        return "Systemprompt (system)"
    if role == "assistant":
        return "Resume-Präfix (assistant, partiell)"
    if content.startswith("<RESULTS>"):
        return "RESULTS-Nachricht (user; fachlich Server-Nachricht)"
    if index == 1:
        return "Frage (user)"
    return "Korrektur-Prompt (user)"


def published_ids(record):
    """Id the harness assigned to each stage's intercepted call (None if nothing was published)."""
    counters, ids = {}, []
    for turn in record["turns"]:
        interrupt = (turn.get("result") or {}).get("interrupt")
        if interrupt and interrupt.get("value") is not None:
            ids.append(harness.assign_id(counters, interrupt["tool"]["name"]))
        else:
            ids.append(None)
    return ids


def render_run(record):
    commit = str(record["git_commit"])
    if record["git_dirty"]:
        files = record.get("git_dirty_files")
        commit += " (getrackte Dateien geändert" + (": " + ", ".join(files) if files else "") + ")"
    head = [
        ("Zeit (UTC)", record["started_at"]),
        ("Experiment", record["experiment"]),
        ("Phase", f"−{record['phase']}"),
        ("Modell", record["model"]),
        ("Endpoint", record["endpoint"]),
        ("Temperatur", record["temperature"]),
        ("max_tokens", record["max_tokens"]),
        ("seed", record["seed"]),
        ("Prompt", f"{record['prompt_version']} · {record['prompt_chars']} Zeichen · sha256 {record['prompt_sha256']}"),
        ("Git-Commit", commit),
        ("Harness sha256", record["harness_sha256"]),
        ("Aufgabe", record["question"]),
        ("Erwartet", json.dumps(record["expected"], ensure_ascii=False)),
        ("Notiz", record["note"]),
        ("Dauer (s)", record["duration_s"]),
        ("Fehler", record["error"]),
    ]
    parts = [f"<h3>Lauf {html.escape(record['run_id'])}</h3>", _table(head)]
    for turn, tag_id in zip(record["turns"], published_ids(record)):
        params = turn["params"]
        parts.append(f"<h4>Stage {turn['stage']}</h4>")
        parts.append(
            f"<p><b>Eingabe</b> (stop_at_tool={params['stop_at_tool']}, dep_value={params['dep_value']}, "
            f"payload_extras={html.escape(json.dumps(turn['payload_extras']))})</p>"
        )
        for index, message in enumerate(turn["messages"]):
            label = html.escape(message_label(index, message))
            is_logged_prompt = message["role"] == "system" and sha256_text(message["content"]) == record["prompt_sha256"]
            if is_logged_prompt:
                parts.append(f"<p>{label}: <i>{html.escape(record['prompt_version'])}, Volltext siehe Abschnitt 1.4</i></p>")
            else:
                parts.append(f"<p>{label}:</p>" + _pre(message["content"]))
        parts.append("<p><b>Ausgabe</b></p>")
        result = turn["result"]
        if result is None:
            parts.append(_table([("Fehler", turn["error"]), ("Dauer (s)", turn["duration_s"])]))
            continue
        parts.append("<p>Thinking (Reasoning-Kanal, wie empfangen):</p>" + _pre(result["reasoning_all"]))
        parts.append("<p>Antwort (Antwortkanal, wie empfangen):</p>" + _pre(result["content_all"]))
        rows = []
        interrupt = result["interrupt"]
        if interrupt is None:
            rows.append(("Abgefangener Tag", "(keiner)"))
        else:
            value = interrupt["value"]
            rows.append(("Abgefangener Tag", interrupt["tool"]["raw"]))
            rows.append(("Kanal", "Antwortkanal (Leak)" if interrupt["leak"] else "Thinking"))
            rows.append(("Wert (Server)", "(nicht auflösbar)" if value is None else harness._fmt(value)))
            rows.append(("Id", tag_id))
            rows.append(("Verworfener Rest", json.dumps(result["discarded"], ensure_ascii=False)))
        rows.append(("finish_reason", result["finish_reason"]))
        rows.append(("Usage", json.dumps(result["usage"]) if result["usage"] else "(keine)"))
        rows.append(("Dauer (s)", turn["duration_s"]))
        parts.append(_table(rows))
    parts.append("<details><summary>Harness-Log (stdout von run_phase)</summary>" + _pre(record["harness_log"]) + "</details>")
    return "\n".join(parts)


def show_run(record):
    display(HTML(render_run(record)))

### 1.8 Offline-Selbsttest (kein Request, kein echter Key)

Ein Mock für `urllib.request.urlopen` spielt die in `doc/instantrag.md` Appendix A.1 dokumentierten Ausgaben
der Phase −3 als SSE-Stream ab. Der Lauf geht durch das unveränderte `run_phase` und den Rekorder und wird nach
`results/selftest.jsonl` geschrieben. Der API-Key ist für die Dauer des Tests ein Dummy.

**Synthetischer Zusatz (nicht aus A.1):** Nach dem Tag in Stage 1 und 2 hängt der Mock ein markiertes Stück an.
Es darf nie gelesen werden – damit wird geprüft, dass der Stream am Tag endet.

Der Selbsttest prüft Mitschreiben, JSONL und Darstellung. Er sagt nichts über das Verhalten des Modells aus.

In [ ]:
import os

SELFTEST_TOKEN = "selftest-dummy-token"
SELFTEST_SEED = 12345
SELFTEST_TAIL = " [MOCK-TAIL: synthetic, must never be read]"
A1_QUESTION = "Please calculate 10+(2*4)"
A1_STAGES = [
    {"reasoning": 'I must obtain <MUL x1="2" x2="4"/>', "content": "", "tail": True},
    {"reasoning": 'I must obtain <MUL id="mul-1"/> and then obtain <ADD x1="10" x2="8"/>', "content": "", "tail": True},
    {
        "reasoning": "",
        "content": "\n\nHere is your result: 10 + (2 * 4) = 18",
        "tail": False,
        "usage": {
            "completion_tokens": 53,
            "prompt_tokens": 797,
            "total_tokens": 850,
            "completion_tokens_details": {"reasoning_tokens": 0, "text_tokens": 53},
        },
    },
]
# inputs of stages 2 and 3 after system prompt and question, verbatim from Appendix A.1
A1_INPUT_TAILS = [
    [],
    [
        {"role": "user", "content": '<RESULTS><MUL id="mul-1">8</MUL></RESULTS>'},
        {"role": "assistant", "content": '<|im_start|>I must obtain <MUL id="mul-1"/>'},
    ],
    [
        {"role": "user", "content": '<RESULTS><MUL id="mul-1">8</MUL> <ADD id="add-1">18</ADD></RESULTS>'},
        {"role": "assistant", "content": '<|im_start|>I must obtain <MUL id="mul-1"/> and then obtain <ADD id="add-1"/>'},
    ],
]


def sse_lines(stage, chunk_size=5):
    """Build the SSE byte lines of one scripted stage (small chunks, like a token stream)."""

    def pieces(text):
        return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

    def event(delta, finish_reason=None):
        return {"choices": [{"index": 0, "delta": delta, "finish_reason": finish_reason}]}

    events = [event({"reasoning_content": piece}) for piece in pieces(stage["reasoning"])]
    if stage["tail"]:
        events.append(event({"reasoning_content": SELFTEST_TAIL}))
    events += [event({"content": piece}) for piece in pieces(stage["content"])]
    events.append(event({}, finish_reason="stop"))
    if stage.get("usage"):
        events.append({"choices": [], "usage": stage["usage"]})
    lines = []
    for item in events:
        lines.append(("data: " + json.dumps(item) + "\n").encode("utf-8"))
        lines.append(b"\n")
    lines.append(b"data: [DONE]\n")
    return lines


class FakeResponse:
    def __init__(self, lines):
        self.lines = lines
        self.consumed = 0

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        return False

    def __iter__(self):
        for line in self.lines:
            self.consumed += 1
            yield line


class FakeUrlopen:
    """Stands in for urllib.request.urlopen: no network, one scripted stream per call."""

    def __init__(self, stages):
        self.stages = stages
        self.calls = []

    def __call__(self, request, timeout=None):
        index = len(self.calls)
        assert index < len(self.stages), "more requests than scripted stages"
        response = FakeResponse(sse_lines(self.stages[index]))
        self.calls.append({
            "url": request.full_url,
            "body": json.loads(request.data.decode("utf-8")),
            # only the comparison result is kept, never the header itself
            "dummy_token_used": request.get_header("Authorization") == "Bearer " + SELFTEST_TOKEN,
            "response": response,
        })
        return response


def run_selftest():
    assert SELFTEST_PATH != RUNS_PATH
    runs_size_before = RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None
    lines_before = len(read_jsonl(SELFTEST_PATH))
    original_run_turn = harness.run_turn
    original_urlopen = harness.urllib.request.urlopen
    key_env = json.loads(SETTINGS_PATH.read_text(encoding="utf-8-sig")).get("api_key_env", "LLM_API_KEY")
    prompt = load_prompt("S0")
    fake = FakeUrlopen(A1_STAGES)

    with mock.patch.dict(os.environ, {key_env: SELFTEST_TOKEN}), \
            mock.patch.object(harness.urllib.request, "urlopen", fake):
        cfg = load_cfg(REFERENCE_MODEL)
        assert cfg["token"] == SELFTEST_TOKEN, "self-test must run with the dummy token"
        assert SELFTEST_TOKEN not in repr(cfg) and SELFTEST_TOKEN not in str(cfg)
        record = run_logged(
            cfg, prompt, A1_QUESTION, 3,
            experiment="selftest",
            seed=SELFTEST_SEED,
            out_path=SELFTEST_PATH,
            expected={"answer": 18, "calls": [["MUL", [2, 4]], ["ADD", [10, 8]]]},
            note="offline self-test: mocked SSE stream replaying doc/instantrag.md Appendix A.1",
        )

    turns = record["turns"]
    # patches are gone again
    assert harness.run_turn is original_run_turn
    assert harness.urllib.request.urlopen is original_urlopen
    assert harness.PAYLOAD_EXTRAS == {}
    # run completed with exactly the three scripted requests
    assert record["error"] is None, record["error"]
    assert len(fake.calls) == 3 and len(turns) == 3
    base = [{"role": "system", "content": prompt["text"]}, {"role": "user", "content": A1_QUESTION}]
    for call, turn, input_tail in zip(fake.calls, turns, A1_INPUT_TAILS):
        assert call["dummy_token_used"]
        assert call["url"].endswith("/chat/completions")
        body = call["body"]
        assert body["seed"] == SELFTEST_SEED and body["model"] == REFERENCE_MODEL
        assert body["temperature"] == TEMPERATURE and body["max_tokens"] == MAX_TOKENS and body["stream"] is True
        # recorded messages are what was sent, and what Appendix A.1 documents
        assert turn["messages"] == body["messages"] == base + input_tail
        assert turn["payload_extras"] == {"seed": SELFTEST_SEED}
        assert turn["error"] is None
    # interceptions, values and final answer as documented
    first, second, third = (turn["result"] for turn in turns)
    assert first["interrupt"]["tool"]["raw"] == '<MUL x1="2" x2="4"/>' and first["interrupt"]["value"] == 8
    assert second["interrupt"]["tool"]["raw"] == '<ADD x1="10" x2="8"/>' and second["interrupt"]["value"] == 18
    assert not first["interrupt"]["leak"] and not second["interrupt"]["leak"]
    assert [turn["params"]["stop_at_tool"] for turn in turns] == [True, True, False]
    assert third["interrupt"] is None and third["finish_reason"] == "stop"
    assert third["content_all"] == A1_STAGES[2]["content"] and third["reasoning_all"] == ""
    assert third["usage"] == A1_STAGES[2]["usage"]
    assert published_ids(record) == ["mul-1", "add-1", None]
    # the stream was stopped at the tag: the synthetic tail was never read
    for call in fake.calls[:2]:
        assert call["response"].consumed < len(call["response"].lines)
    assert fake.calls[2]["response"].consumed == len(fake.calls[2]["response"].lines)
    assert SELFTEST_TAIL not in json.dumps(record)
    # metadata
    assert record["schema_version"] == SCHEMA_VERSION
    assert record["prompt_sha256"] == sha256_text(harness.load_system_prompt(PROMPT_PATHS["S0"]))
    assert record["git_commit"] == git_commit() and record["harness_sha256"] == harness_sha256()
    assert record["git_dirty_files"] == git_dirty_files()
    assert record["git_dirty"] == bool(record["git_dirty_files"])
    assert "[guard] forged tool results in model output : none" in record["harness_log"]
    # JSONL: exactly one line appended, readable, identical, no token
    stored = read_jsonl(SELFTEST_PATH)
    assert len(stored) == lines_before + 1
    assert stored[-1] == json.loads(json.dumps(record))
    assert SELFTEST_TOKEN not in SELFTEST_PATH.read_text(encoding="utf-8")
    assert (RUNS_PATH.stat().st_size if RUNS_PATH.exists() else None) == runs_size_before
    # display renders and keeps tags visible
    rendered = render_run(record)
    assert html.escape('<MUL x1="2" x2="4"/>') in rendered
    assert html.escape('<|im_start|>I must obtain <MUL id="mul-1"/>') in rendered
    print(f"SELFTEST OK: run {record['run_id']}, 3 mocked requests, "
          f"{SELFTEST_PATH.relative_to(ROOT).as_posix()} now has {len(stored)} line(s)")
    return record


selftest_record = run_selftest()

**Darstellung des Selbsttest-Laufs** (Daten aus dem Mock, kein Modellverhalten):

In [ ]:
show_run(selftest_record)

### 1.9 Aufgaben

*Platzhalter – Schritt 5:* `config/poc_items.json` mit den eingefrorenen Aufgabenlisten T1/T2, richtigem Ergebnis
und erwarteten Calls.

## 2. V0 – Messung prüfen (a–d)

*Gliederungspunkt – Schritt 4.* (a) `/v1/models`, (b) Cache-Test, (c) Fortsetzungstest, (d) sichtbare Zeichen
vs. `completion_tokens` (Plan §7).

## 3. Phase −1: ein Lauf, vollständig dokumentiert

*Gliederungspunkt.* Stream bis zum ersten leeren Tag, Abbruch; nichts wird zurückgeschickt.

## 4. Phase −2: ein Lauf, vollständig dokumentiert

*Gliederungspunkt.* RESULTS-Nachricht, Resume-Präfix mit Id-Verweis, zweiter Tool-Aufruf.

## 5. Phase −3: Experimente E1–E5

*Gliederungspunkt – ab Schritt 6.* Matrix-Schleife über Prompt, Modell und Aufgaben mit `RUN`-Schalter
(Standard `RUN = False`); Rohdaten nach `results/runs.jsonl`.

## 6. Auswertung

*Gliederungspunkt – Bewertungscodes in Schritt 3.* Codes nach Plan §8, Raten mit Wilson-KI je Zelle, gepaarte
Vergleiche (exakter McNemar), Fehler je Schritt, Sicht „bis Stage 1 / 2 / 3“.

## 7. Laborbuch

*Gliederungspunkt.* Abweichungen, Auffälligkeiten, offene Punkte – mit Datum.

## 8. Interaktive Erkundung (Playground)

Hier lässt sich eine einzelne Frage frei durch den unveränderten Harness schicken. In **Phase −1** läuft nur ein
Request: Der Stream wird beim ersten leeren Tool-Tag (z. B. `<ADD x1="4" x2="9"/>`) abgebrochen, der Harness
rechnet das Ergebnis aus, schickt es aber nicht zurück – man sieht also, ob und welchen ersten Call das Modell
vorschlägt. In **Phase −2** wird dieses Ergebnis in einer `<RESULTS>`-Nachricht veröffentlicht und das
unterbrochene Thinking mit dem Id-Verweis als Resume-Präfix fortgesetzt; man sieht den zweiten Call und ob er den
Serverwert verwendet. **Phase −3** veröffentlicht auch das zweite Ergebnis und fordert dann ohne Tool-Stopp die
Endantwort an, die nur aus Serverwerten stammen sollte. Bleibt ein Tag aus, endet der Lauf mit der Halteantwort;
ab Phase −2 schickt der Harness dann einmal einen Korrektur-Prompt.

Playground-Läufe landen nur in `results/playground.jsonl` (`experiment = "playground"`), nie in
`results/runs.jsonl`, und zählen nicht zu den Experimenten.

In [ ]:
QUESTION = "Please calculate 7*(4+9)"
PHASE = -3                  # -1, -2 or -3
MODEL = REFERENCE_MODEL     # e.g. "qwen-3.8-180b-next"
PROMPT_VERSION = "S0"       # key of PROMPT_PATHS
TEMPERATURE = 0.1
SEED = None                 # None: random seed for this run, stored in the record
RUN = False                 # True sends requests to the LLM endpoint

In [ ]:
import random


def playground_run(question, phase, model, prompt_version, temperature, seed, run,
                   cfg=None, out_path=None):
    """Summarise what would be sent; only with run=True send it via run_logged and return the record."""
    out_path = Path(out_path or PLAYGROUND_PATH)
    if phase not in (-1, -2, -3):
        raise ValueError(f"PHASE must be -1, -2 or -3, got {phase!r}")
    if prompt_version not in PROMPT_PATHS:
        raise ValueError(f"unknown PROMPT_VERSION {prompt_version!r}; available: {sorted(PROMPT_PATHS)}")
    if out_path.resolve() == RUNS_PATH.resolve():
        raise ValueError("the playground never writes to results/runs.jsonl")
    cfg = model_cfg if cfg is None else cfg
    prompt = load_prompt(prompt_version)
    if seed is None:
        seed = random.randrange(2**31)
    print("playground run" + ("" if run else " (RUN = False: nothing is sent)"))
    print(f"  model      : {model}")
    print(f"  endpoint   : {cfg['endpoint'] if cfg is not None else '(no API key loaded)'}")
    print(f"  phase      : {phase}")
    print(f"  question   : {question}")
    print(f"  prompt     : {prompt['version']} ({prompt['path']}, sha256 {prompt['sha256'][:12]}…)")
    print(f"  temperature: {temperature} | max_tokens: {MAX_TOKENS} | seed: {seed}")
    print(f"  output     : {out_path.relative_to(ROOT).as_posix() if out_path.is_relative_to(ROOT) else out_path}")
    if not run:
        print("no request sent; set RUN = True to run")
        return None
    if cfg is None:
        raise RuntimeError("no API key loaded (section 1.3): a live run is not possible")
    run_cfg = SafeCfg(cfg)
    run_cfg["model"] = model
    record = run_logged(
        run_cfg, prompt, question, abs(phase),
        experiment="playground",
        seed=seed,
        out_path=out_path,
        temperature=temperature,
        note="playground: exploratory, not part of the experiment matrix",
    )
    show_run(record)
    return record


playground_record = playground_run(QUESTION, PHASE, MODEL, PROMPT_VERSION, TEMPERATURE, SEED, RUN)

Nach einem Lauf `RUN` wieder auf `False` stellen, damit „Run All“ keine Requests auslöst.